In [0]:
import requests
import urllib3
import pandas as pd
import logging
import os
import time
import json
from datetime import datetime
from pathlib import Path
import configparser


class webi_Document: #WEBI Document Object to hold the structured details for the MetaData extraction
    def __init__(self, document_id: str, document_cuid: str, document_name: str, folder_id: str, full_path: str, updated: str, scheduled: str, created: str, lastAuthor: str):
        self.document_id = document_id
        self.document_cuid = document_cuid
        self.document_name = document_name
        self.folder_id = folder_id
        self.full_path = full_path
        self.updated = updated
        self.scheduled = scheduled
        self.created = created
        self.lastAuthor = lastAuthor
        self.data_providers: list = [] # Container for data providers details
        self.Execution_Time_Sec: float = 0.0
        self.Number_of_Data_Providers: int = 0
        self.Number_of_API_Calls: int = 0
        self.API_Pause_Counts: int = 0
        self.Start_Time: datetime = None
        self.End_Time: datetime = None
        
    def add_data_provider(self, data_provider: dict): # Add details to the WEBI document data provider container, expect details to be in a dictionary format
        if isinstance(data_provider, dict):
            self.data_providers.append(data_provider)
            logging.info(f"Added data provider ID {data_provider.get('Data_Provider_ID')} to WEBI Document ID {self.document_id}.")
        else:
            logging.warning("Data provider should be a dictionary.")
    
    def get_details(self) -> dict: # Get the WEBI document details as a dictionary
        return {
            "Document_Id": self.document_id,
            "Document_CUID": self.document_cuid,
            "Folder_Id": self.folder_id,
            "Full_path": self.full_path,
            "Document_name": self.document_name,
            "updated": self.updated,
            "scheduled": self.scheduled,
            "created": self.created,
            "lastAuthor": self.lastAuthor
        }
    
    def add_Extraction_Output(self, input: dict): # Add the extraction execution details to the WEBI document object
        self.Execution_Time_Sec: float = input.get("Execution_Time_Sec", 0.0)
        self.Number_of_Data_Providers: int = input.get("Number_of_Data_Providers", 0)
        self.Number_of_API_Calls: int = input.get("Number_of_API_Calls", 0)
        self.API_Pause_Counts: int = input.get("API_Pause_Counts", 0)
        self.Start_Time: datetime = input.get("Start_Time", None)
        self.End_Time: datetime = input.get("End_Time", None)
    
    def save_webi_json(self, folder_path: str, today_date: str): # Save the WEBI document details and data providers to a JSON file
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            logging.info(f"Created directory for WEBI Document JSON: {folder_path}")
        # Check if file already exists, if so, append a number to the filename
        file_Name=f"{folder_path}\\WEBI_Document_{self.document_id}_{today_date}.json"
        if os.path.exists(file_Name):
            count = 1
            while True:
                new_file_Name = f"{folder_path}\\WEBI_Document_{self.document_id}_{today_date}({count}).json"
                if not os.path.exists(new_file_Name):
                    file_Name = new_file_Name
                    break
                count += 1

        with open(file_Name, "w", encoding="utf-8") as json_file:
            json.dump({
                "Document_Details": self.get_details(),
                "Extraction_Stats": {
                    "Execution_Time_Sec": self.Execution_Time_Sec,
                    "Number_of_Data_Providers": self.Number_of_Data_Providers,
                    "Number_of_API_Calls": self.Number_of_API_Calls,
                    "API_Pause_Counts": self.API_Pause_Counts,
                    "Start_Time": str(self.Start_Time),
                    "End_Time": str(self.End_Time)
                },
                "Data_Providers": self.data_providers
            }, json_file, ensure_ascii=False, indent=4)

def ensure_file_exists(path: str | Path) -> Path:
    """Ensure the config file exists and return a Path object."""
    p = Path(path).expanduser().resolve()
    if not p.exists():
        raise FileNotFoundError(f"Configuration file not found: {p}")
    return p


def load_json_config(path: str | Path) -> dict:
    p = ensure_file_exists(path)
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def Document_Details(Output: list, doc: dict, dp: dict, DP_Detail: dict, Row_index:str, sql_query: str, WEBI_Doc: webi_Document):
    #Put in WEBI Document Object
    Dataprovider:dict = {
        "Data_Provider_ID": dp.get("id"),
        "Data_Provider_Name": dp.get("name"),
        "Data_Profider_Refresh_Time": dp.get("updated"),
        "Data_Source_ID": dp.get("dataSourceId"),
        "DataSource_Type": dp.get("dataSourceType"),
        "DataSource_Name": DP_Detail.get("dataSourceName"),
        "SQL_Index": Row_index,
        "SQL_Query": sql_query                             
    }
    WEBI_Doc.add_data_provider(Dataprovider)

    logging.info(f"Appending Data Provider ID {dp.get('id')} details to output.")
    dp_output = {
        "Document_Id": doc.get("id"),
        "Document_CUID": doc.get("cuid"),
        "Folder_Id": doc.get("folderId"),
        "Full_path": doc.get("path"),
        "Document_name": doc.get("name"),
        "updated": doc.get("updated"),
        "scheduled": doc.get("scheduled"),
        "created": doc.get("createdBy"),
        "lastAuthor": doc.get("lastAuthor"),
        "Data_Provider_ID": dp.get("id"),
        "Data_Provider_Name": dp.get("name"),
        "Data_Profider_Refresh_Time": dp.get("updated"),
        "Data_Source_ID": dp.get("dataSourceId"),
        "DataSource_Type": dp.get("dataSourceType"),
        "DataSource_Name": DP_Detail.get("dataSourceName"),
        "SQL_Index": Row_index,
        "SQL_Query": sql_query                 
    }
    Output.append(dp_output)

def log_off_call(bo_url: str, logon_token: str, log_status: bool):

    if log_status is True: 
        # Step 6: Log off session
        logoff_url = f"{bo_url}/logoff"
        logoff_headers = {"X-SAP-LogonToken": logon_token}
        requests.post(logoff_url, headers=logoff_headers, verify=False)
        log_status: bool = False
        logging.info("Logged off from BO REST API.")

def API_Json_CAll(API_url: str, headers: dict, Todyay_Date: str, API_CALLS: dict, FolderPath: str, document) -> dict:
    # logging.info(f"API URL Used for WEBI Document DataProvider Details: {API_url}")
    API_response_TimeFrame=5
    API_Retry_Attempt=3
    while API_Retry_Attempt >0:
        Before_API_Call=datetime.now()
        logging.info(f"API URL Used for WEBI Document: {API_url}")
        API_responded: str = ""
        API_url_CALL = requests.get(API_url, headers=headers, verify=False)
        API_CALLS["Total Calls"] += 1
        API_CALLS["Document Calls"] += 1
        After_API_Call=datetime.now()
        Respond_Time=(After_API_Call - Before_API_Call).total_seconds()
        if Respond_Time> API_response_TimeFrame:
            logging.warning(f"Data Provider list API call for Data Providers took {Respond_Time} seconds. Paulse for {API_response_TimeFrame} seconds.")
            time.sleep(5)  # Pause for 5 seconds before proceeding
            API_CALLS["Pauses"] += 1
        if API_url_CALL.status_code == 200:
            API_Retry_Attempt = 0
            try:
                API_responded = API_url_CALL.json()
                Save_API_Outputs(FolderPath, API_responded, Todyay_Date, document, API_url, API_responded)
            except ValueError:
                logging.error(f"Response is not valid JSON for API URL: {API_url}")
            except Exception as e:
                logging.error(f"Error extracting Json: {e}")
        elif API_url_CALL.status_code == 404:
            logging.error(f"Error {API_url_CALL.status_code} FOR API Call. Retrying...{API_Retry_Attempt} attempts left. Waiting for 10 seconds before retrying.")
            API_Retry_Attempt -= 1
            time.sleep(10)  # Wait before retrying
        else:
            API_responded = API_url_CALL.json()
            logging.error(f"Error {API_url_CALL.status_code} FOR API Call. with response: {API_responded} ")
            API_Retry_Attempt=0


    return API_responded

def Save_Outputs(FolderPath: str, Output: list, DOC_Output: list, Todyay_Date: str):
    if Output and DOC_Output:
        try:
            SQLOUTPUT_DF = pd.DataFrame(Output)
            Execution_DF = pd.DataFrame(DOC_Output)
            # Save Extraction output
            # print(FolderPath)
            FileNames={
                "SQL_csv_file":f"{FolderPath}\\WEBI_DataProviders_SQL_{Todyay_Date}.csv",
                "Execution__csv_file": f"{FolderPath}\\WEBI_Execution_Stats_{Todyay_Date}.csv"
            }
            for key, FileName in FileNames.items():
                if os.path.exists(FileName):
                    count = 1
                    while True:
                        new_file_Name = FileName.replace(".csv", f"({count}).csv")
                        if not os.path.exists(new_file_Name):
                            FileNames[key] = new_file_Name
                            break
                        count += 1 
            SQLOUTPUT_DF.to_csv(FileNames["SQL_csv_file"], index=False, quoting=1)
            Execution_DF.to_csv(FileNames["Execution__csv_file"], index=False, quoting=1)
            logging.info(f"Outputs saved successfully to {FolderPath}\\WEBI_DataProviders_SQL_{Todyay_Date}.csv")
        except Exception as e:
            logging.error(f"An error occurred while saving outputs: {e}")   

def Save_API_Outputs(FolderPath: str, API_Output: dict, Todyay_Date: str, document: str, API_url: str, API_responded: dict):
    FolderPath_1 = FolderPath + f"\\API_Response_{Todyay_Date}"
    if not os.path.exists(FolderPath_1):
        os.makedirs(FolderPath_1)
        logging.info(f"Created directory for API responses: {FolderPath_1}")
    FolderPath_2 = FolderPath_1 + f"\\API_Response_For_{document}"
    if not os.path.exists(FolderPath_2):
        os.makedirs(FolderPath_2)
        logging.info(f"Created directory for API responses: {FolderPath_2}")
    # Save the API response to a JSON file
    api_filename = API_url.split("/")[-1]
    if api_filename == document:
        api_filename = f"{document}_Details"
    else:
        if api_filename == "queryplan":
            DataProvider_ID=API_url.split("/")[-2]
            api_filename = f"{document}_{DataProvider_ID}_{api_filename}_Response"
        else:
            api_filename = f"{document}_{api_filename}_Response"
    file_Name=f"{FolderPath_2}\\{api_filename}.json"
    if os.path.exists(file_Name):
        count = 1
        while True:
            new_file_Name = f"{FolderPath_2}\\{api_filename}({count}).json"
            if not os.path.exists(new_file_Name):
                file_Name = new_file_Name
                break
            count += 1  
    with open(file_Name, "w", encoding="utf-8") as json_file:
        json.dump(API_responded, json_file, ensure_ascii=False, indent=4)


def loop_Dict(Raw_data) -> list: #, parent_key='', sep='_'
    Output: list = []
    # keys = list(Raw_data.keys())
    index:str=""
    sql_query:str=""
    for key, values in Raw_data.items():
        if isinstance(values, dict):
            Output.extend(loop_Dict(values))
        elif isinstance(values, list):
            Output.extend(loop_List(values))
        else:
            match key:
                case "@index": 
                    index=values
                case "$": 
                    sql_query = values
    if index!="" and sql_query!="":
        Output.append({
            "index": index,
            "sql_query": sql_query
        })

    return Output

def loop_List(Raw_data) -> list: #, parent_key='', sep='_'
    Output: list = []
    index:str=""
    sql_query:str=""
    for prop in Raw_data:
        
        if type(prop) is dict:
            #change below to Raw_data[prop]
            if prop.get("statement",[]):
                Output.extend(loop_List(prop.get("statement",[])))
            elif prop.get("statement",{}):
                    Output.extend(loop_List(prop.get("statement",{})))
            else:
                    Output.extend(loop_List(prop))

            continue
        match prop:
            case "@index": 
                index=Raw_data[prop]
            case "$": 
                sql_query = Raw_data[prop]
    if index!="" and sql_query!="":
        Output.append({
            "index": index,
            "sql_query": sql_query
        })
    return Output


def main():
    # Disable SSL warnings (only for testing)
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    Todyay_Date = datetime.now().strftime("%Y-%m-%d")
    logging.basicConfig(
        filename=f"Script_Dev_{Todyay_Date}.log",          # Dev Test Log file name
        # filename='Script_Prod_MB.log',          #Prod Log file name
        level=logging.INFO,          # Log level: DEBUG, INFO, WARNING, ERROR, CRITICAL
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    logging.info(f"Script started: {Todyay_Date} with directory {os.getcwd()}")
    Current_path=os.getcwd()
    config_file =Current_path+ "/config.json"
    ensure_file_exists(config_file)
    config = load_json_config(config_file)
    logging.info(f"Config file loaded: {config_file}")
    SAP_BO_Dict=config.get('SAP_BO_Credentials',{})
    fileName=config.get('Static_Dict',{}).get('Report_ID_List')
    Report_fileName=Current_path+"/"+fileName
    logging.info(f"Report File Name: {Report_fileName}")
    # Read the tab-delimited file
    ensure_file_exists(Report_fileName)
    webi_list_df = pd.read_csv(Report_fileName, delimiter='\t')  # Use '\t' for tab-delimited files
    logging.info(f"Number of records in the file: {webi_list_df.count()}")
    

    #Prepare for the Scan
    DOC_Output = []
    Output = []
    API_CALLS = {"Total Calls":0, "Document Calls":0, "Pauses":0}
    log_status: bool = False
    # Step 1: Log in and get token
    try:
        bo_url = SAP_BO_Dict.get("bo_url")
        logon_url = f"{bo_url}logon/long"
        logon_payload = {
            "userName": SAP_BO_Dict.get("username"),
            "password": SAP_BO_Dict.get("password"),
            "auth": SAP_BO_Dict.get("auth_type")
        }
        headers = {"Content-Type": "application/json"}
        logging.info(f"API URL Used for Logon: {logon_url}")
        response = requests.post(logon_url, json=logon_payload, headers=headers, verify=False)
        API_CALLS["Total Calls"] += 1
        logon_token = response.headers.get("X-SAP-LogonToken", None)
        logging.info(f"Logon Token:{logon_token}")
        print(f"Logon Token:{logon_token}")
        log_status = True if logon_token else False
    except Exception as e:
        logging.error(f"Error occurred while logging in: {e}")

    if log_status is True:
        logging.info("Successfully logged in to BO REST API.")



    
main()